# FinExpert v2 — real-report data (FinQA + TAT-QA) + synthetic, QLoRA on Qwen3-4B

Runs on **Kaggle** (recommended) or **Colab**; the notebook detects which.

| Kaggle setup (once) | |
|---|---|
| Phone-verify your account | required for GPUs |
| Settings → Accelerator → **GPU T4 x2** | not P100 |
| Settings → **Internet on** | for pip, the repo and the model |
| Real runs: **Save Version → Save & Run All** | runs in the background; outputs kept in the version's Output tab |

**Three modes** (set `RUN_MODE` in the config cell):

| Mode | What it does | Typical time on a T4 |
|---|---|---|
| `train` | Fine-tunes on the committed mix file (template targets, or distilled reasoning targets if `DISTILLED_TARGETS` is set), then evaluates | ~1–1.5 h for the pilot mix |
| `baseline` | No training: evaluates the *base* model with the prompt variants in `BASELINE_VARIANTS` | ~1.5–2.5 h per variant |
| `distill` | No training: the base model answers the mix's *training* questions; answers verified correct become reasoning targets | ~3.5 h |
| `test` | **Final evaluation only (v3):** loads a saved, already-validated adapter and scores the official FinQA / TAT-QA **test** sets once | ~2–6 h depending on answer style |

All modes rebuild the FinQA/TAT-QA data from the repo, verify it byte-for-byte against `audit.json`, and score the **same**
600 validation questions + 30 synthetic test examples with the **same** scoring code, so their numbers are directly comparable.

## 1. Config (only edit this cell)

In [ ]:
RUN_MODE   = "train"                                # "train", "baseline", "distill" or "test"
SEED       = 3407
# Baseline mode: which prompt variants to evaluate (each ~1.5-2.5 h on a T4).
#   zero_shot, few_shot                 -> recorded in experiments/base-zero-few-shot
#   zero_shot_rules, few_shot_rules     -> same, plus the QA answer-format rules (scale words, "; " between items)
BASELINE_VARIANTS = ["zero_shot_rules"]
# Train mode: None = template targets (v2-A); a committed distill output = reasoning targets (v2-B)
DISTILLED_TARGETS = None       # v2-B: "data/external/distilled/distilled_targets_v2_pilot.jsonl"
# Test mode (v3): which validated run's adapter to load, and which official test sets to score.
# On Kaggle: Add Input -> Your Work -> the notebook version that trained ADAPTER_RUN (its output is mounted read-only).
ADAPTER_RUN  = "v2-pilot-seed42"
TEST_SOURCES = ["finqa", "tatqa"]   # split across two runs if needed, e.g. ["finqa"] and ["tatqa"]
RUN_NAME   = {"train": f"v2{'b' if DISTILLED_TARGETS else ''}-pilot-seed{SEED}",
              "test": f"test-{ADAPTER_RUN}-" + "-".join(TEST_SOURCES),
              "baseline": "base-" + "-".join(v.replace("_", "-") for v in BASELINE_VARIANTS),
              "distill": "distill-v2-pilot"}[RUN_MODE]
REPO_URL   = "https://github.com/mani1234765/finexpert.git"
REPO_REF   = "main"                                  # exact commit is recorded
MIX_FILE   = "data/external/train_mix_v2_pilot.json" # which examples to train on
EVAL_SUBSET = "data/external/eval_subset_v1.json"    # fixed validation questions (never train on these)

# v1 synthetic data (unchanged; hash-checked)
SYNTH_SHA256 = {
    "train":      "b9fe27b8981ed74ef5525852730c4f213578334ea1aec6240b708ad2968baa63",
    "validation": "165c3bf024ef6f2b3f9140116216d570ec4f7ccdb7a0c04e3264a076cc336a84",
    "test":       "851220d99e6d33cfab95334ad3ee9729452860f8aac1a04f5feb22032d5ae33d",
}

# Model + LoRA: same as v1.1. EPOCHS is 1 for the pilot (4x more data than v1).
MODEL_NAME     = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 16, 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EPOCHS         = 1
LEARNING_RATE  = 2e-4
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
WARMUP_RATIO   = 0.05
LR_SCHEDULER   = "linear"
WEIGHT_DECAY   = 0.01
EVAL_EVERY_STEPS = 25          # validation loss on the monitoring set (not used to pick checkpoints)

# Generation (fine-tuned model)
GEN_BATCH_SIZE      = 8
MAX_NEW_TOKENS_QA   = 1024 if DISTILLED_TARGETS else 384   # reasoning targets make answers longer; hits are flagged
MAX_NEW_TOKENS_SYNTH = 700

# Baseline mode (base model, no training)
MAX_NEW_TOKENS_BASE = 1024     # base models write longer answers; a low limit would unfairly cut them off
BASE_MAX_SEQ_LENGTH = 8192     # few-shot prompts are longer than training examples
STOP_AFTER_ANSWER   = True     # stop a base-model answer once its "Answer: ..." line is complete

UNSLOTH_VERSION = None         # pin after the first run (see run_manifest.json)
COLAB_USE_DRIVE = False        # Colab only: copy the run folder to Drive

## 2. Platform, GPU, install

In [ ]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle T4 x2: train on one GPU (Unsloth is single-GPU)
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB  = "google.colab" in sys.modules
PLATFORM  = "kaggle" if IN_KAGGLE else ("colab" if IN_COLAB else "local")
WORK_DIR  = "/kaggle/working" if IN_KAGGLE else ("/content" if IN_COLAB else os.getcwd())
REPO_DIR  = "/tmp/finexpert-repo" if IN_KAGGLE else os.path.join(WORK_DIR, "finexpert-repo")  # not saved as output
RUN_DIR   = os.path.join(WORK_DIR, "runs", RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

import torch
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > T4 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
assert "P100" not in GPU_NAME, "P100 selected; switch the accelerator to GPU T4 x2"
print(PLATFORM, "|", GPU_NAME, "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB | run dir", RUN_DIR)

In [ ]:
import subprocess
def sh(*cmd, cwd=None):
    r = subprocess.run(list(cmd), capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return r.stdout.strip()

try:
    sh("git", "ls-remote", REPO_URL, "HEAD")
except RuntimeError:
    raise RuntimeError("No internet. Kaggle: Settings > Internet on (needs phone verification).")

sh(sys.executable, "-m", "pip", "install", "-q", "unsloth" if UNSLOTH_VERSION is None else f"unsloth=={UNSLOTH_VERSION}")
print("Installed unsloth")

## 3. Repo, evaluators, and a byte-for-byte rebuild of the external data

In [ ]:
import json, hashlib
def git(*a): return sh("git", "-C", REPO_DIR, *a)
if not os.path.exists(REPO_DIR):
    sh("git", "clone", "-q", REPO_URL, REPO_DIR)
git("fetch", "-q", "--all", "--tags")
_is_branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "-q", f"origin/{REPO_REF}"],
                            capture_output=True).returncode == 0
git("checkout", "-q", "--detach", f"origin/{REPO_REF}" if _is_branch else REPO_REF)
REPO_COMMIT = git("rev-parse", "HEAD")
assert git("status", "--porcelain", "--untracked-files=no") == "", "repo has local changes; delete REPO_DIR and rerun"
print("Repo commit:", REPO_COMMIT)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
for m in [m for m in list(sys.modules) if m.startswith("finexpert")]:
    del sys.modules[m]
from finexpert.evaluation.report import build_report, format_report
from finexpert.evaluation.qa.report import build_qa_report, format_qa_report
from finexpert.data.external.common import SYSTEM_PROMPT
from finexpert.evaluation.prompting import (answer_line_complete, build_messages, select_few_shot, source_key,
                                            to_chatml, training_text)
from finexpert.evaluation.qa.report import score_record
from finexpert.data.distill import apply_targets, build_distilled_targets
print("Evaluators and prompt builder loaded")

In [ ]:
# Clone FinQA / TAT-QA at the exact commits recorded in audit.json, rebuild, and verify every file hash.
P = lambda *a: os.path.join(REPO_DIR, *a)
audit = json.load(open(P("data/external/audit.json")))
for name, folder in [("finqa", "FinQA"), ("tatqa", "TAT-QA")]:
    src = audit["sources"][name]
    dest = P("data/external/raw", folder)
    url = {"finqa": "https://github.com/czyssrs/FinQA.git", "tatqa": "https://github.com/NExTplusplus/TAT-QA.git"}[name]
    if not os.path.exists(dest):
        sh("git", "clone", "-q", url, dest)
    sh("git", "-C", dest, "checkout", "-q", src["source_commit"])
sh(sys.executable, "scripts/build_external_dataset.py", cwd=REPO_DIR)

def sha(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()
for name, src in audit["sources"].items():
    for fname, expected in src["sha256"].items():
        assert sha(P("data/external", name, fname)) == expected, f"{name}/{fname} differs from audit.json"
assert git("status", "--porcelain", "--untracked-files=no") == "", "rebuild changed tracked files"
for split, expected in SYNTH_SHA256.items():
    assert sha(P("data/sft", f"{split}.jsonl")) == expected, f"synthetic {split} hash mismatch"
print("External data rebuilt and verified against audit.json; synthetic data verified.")

## 4. Load the training mix, monitoring set and evaluation sets

In [ ]:
from collections import Counter
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

mix = json.load(open(P(MIX_FILE)))
subset = json.load(open(P(EVAL_SUBSET)))
eval_ids = {i for s in subset["sources"].values() for i in s["example_ids"]}

train_records, monitor_records, qa_eval_records = [], [], []
for source, spec in mix["sources"].items():
    assert sha(P("data/external", source, "train.jsonl")) == spec["train_sha256"], f"{source} train changed since the mix was made"
    by_id = {r["example_id"]: r for r in read_jsonl(P("data/external", source, "train.jsonl"))}
    train_records += [by_id[i] for i in spec["train_ids"]]
    val = {r["example_id"]: r for r in read_jsonl(P("data/external", source, "validation.jsonl"))}
    monitor_records += [val[i] for i in spec["monitor_ids"]]
    qa_eval_records += [r for i, r in val.items() if i in eval_ids]

synth = {s: read_jsonl(P("data/sft", f"{s}.jsonl")) for s in SYNTH_SHA256}
assert sha(P(mix["synthetic"]["path"])) == mix["synthetic"]["sha256"]
train_records += synth["train"]
monitor_records += synth["validation"]

train_ids = {r["example_id"] for r in train_records}
assert not train_ids & eval_ids, "LEAK: training examples overlap the evaluation subset"
assert not train_ids & {r["example_id"] for r in monitor_records}, "LEAK: train/monitor overlap"
assert len(qa_eval_records) == len(eval_ids), "evaluation subset incomplete"
print("train  ", len(train_records), dict(Counter(r.get("source_type", "synthetic") for r in train_records)))
print("monitor", len(monitor_records), "| QA eval", len(qa_eval_records), "| synthetic test", len(synth["test"]))

# Few-shot worked examples: chosen from the FULL training splits (not just the mix), deterministically.
few_shot_pool = (read_jsonl(P("data/external/finqa/train.jsonl")) + read_jsonl(P("data/external/tatqa/train.jsonl"))
                 + synth["train"])
SHOTS = select_few_shot(few_shot_pool)
shot_ids = {e["example_id"] for v in SHOTS.values() for e in v}
assert not shot_ids & (eval_ids | {r["example_id"] for r in synth["test"]}), "LEAK: a worked example is an eval question"
print("few-shot examples:", {k: [e["example_id"] for e in v] for k, v in SHOTS.items()})

# v3: official test sets (scored once, for the selected model only)
test_records = []
if RUN_MODE == "test":
    for source in TEST_SOURCES:
        test_records += read_jsonl(P("data/external", source, "test.jsonl"))
    assert not {r["example_id"] for r in test_records} & {r["example_id"] for r in train_records}, "LEAK: test in train"
    print("official test questions:", len(test_records), dict(Counter(r["source_type"] for r in test_records)))

# v2-B: swap in the distilled reasoning targets (accepted examples only; the rest keep template targets)
DISTILLED_INFO = None
if RUN_MODE == "train" and DISTILLED_TARGETS:
    rows = read_jsonl(P(DISTILLED_TARGETS))
    qa_train_ids = {r["example_id"] for r in train_records if r.get("source_type") in ("finqa", "tatqa")}
    assert {r["example_id"] for r in rows} == qa_train_ids, "distilled file does not match this mix's QA questions"
    train_records = apply_targets(train_records, rows)
    DISTILLED_INFO = {"path": DISTILLED_TARGETS, "sha256": sha(P(DISTILLED_TARGETS)),
                      "accepted": sum(r["accepted"] for r in rows), "total": len(rows)}
    print(f"Distilled targets applied: {DISTILLED_INFO['accepted']}/{DISTILLED_INFO['total']} QA examples now train on reasoning")

## 5. Load the model

* `train` mode: the 4-bit base model plus fresh LoRA adapters (the only weights that train).
* `baseline` mode: the 4-bit base model alone, with a longer context window because few-shot prompts are longer.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
if RUN_MODE == "train":
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
                                                         dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(model, r=LORA_R, target_modules=TARGET_MODULES, lora_alpha=LORA_ALPHA,
                                             lora_dropout=LORA_DROPOUT, bias="none",
                                             use_gradient_checkpointing="unsloth", random_state=SEED)
    model.print_trainable_parameters()
elif RUN_MODE == "test":
    import glob
    search_root = "/kaggle/input" if IN_KAGGLE else WORK_DIR
    found = sorted({os.path.dirname(p) for p in glob.glob(f"{search_root}/**/{ADAPTER_RUN}/adapter/adapter_config.json",
                                                           recursive=True)})
    assert len(found) == 1, (f"expected one adapter for {ADAPTER_RUN} under {search_root}, found {found}. "
                             "Kaggle: Add Input -> Your Work -> the notebook version that trained it.")
    ADAPTER_DIR = found[0]
    adapter_cfg = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json")))
    assert adapter_cfg["base_model_name_or_path"] == MODEL_NAME, adapter_cfg["base_model_name_or_path"]
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=BASE_MAX_SEQ_LENGTH,
                                                         dtype=None, load_in_4bit=True)
    REASONING_STYLE = ADAPTER_RUN.startswith("v2b")
    print("Loaded adapter", ADAPTER_DIR, "| r =", adapter_cfg["r"], "alpha =", adapter_cfg["lora_alpha"],
          "| reasoning-style answers:", REASONING_STYLE)
elif RUN_MODE in ("baseline", "distill"):
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=BASE_MAX_SEQ_LENGTH,
                                                         dtype=None, load_in_4bit=True)
    print("Base model loaded (no adapters)")
else:
    raise ValueError(f"RUN_MODE must be 'train', 'baseline', 'distill' or 'test', not {RUN_MODE!r}")

## 6. Train (`train` mode only)

Training text comes from `finexpert.evaluation.prompting.training_text`, the same function the evaluation prompts are built with,
so training and evaluation prompts are byte-identical (a unit test in the repo guards this). Loss is computed on the assistant
answer only: the model is graded on what it writes, not on copying the question.

In [ ]:
import time
TRAIN_DROPPED, TRAIN_TOKENS, TRAIN_MINUTES, TOKENS_PER_SEC, PEAK_VRAM_GB, log = [], None, None, None, None, []
if RUN_MODE == "train":
    import inspect
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only

    def fit(records, name):
        lengths = {r["example_id"]: len(tokenizer(training_text(r)).input_ids) for r in records}
        kept = [r for r in records if lengths[r["example_id"]] <= MAX_SEQ_LENGTH]
        dropped = [r["example_id"] for r in records if lengths[r["example_id"]] > MAX_SEQ_LENGTH]
        total = sum(lengths[r["example_id"]] for r in kept)
        print(f"{name}: kept {len(kept)}, dropped {len(dropped)} over {MAX_SEQ_LENGTH} tokens | "
              f"mean {total / len(kept):.0f}, max {max(lengths[r['example_id']] for r in kept)}, total {total:,} tokens")
        return kept, dropped, total

    train_fit, TRAIN_DROPPED, TRAIN_TOKENS = fit(train_records, "train")
    monitor_fit, _, _ = fit(monitor_records, "monitor")
    train_ds = Dataset.from_list([{"text": training_text(r)} for r in train_fit]).shuffle(seed=SEED)
    monitor_ds = Dataset.from_list([{"text": training_text(r)} for r in monitor_fit])
    assert all("<think>" not in t for t in train_ds["text"])

    _len_key = "max_length" if "max_length" in inspect.signature(SFTConfig.__init__).parameters else "max_seq_length"
    args = SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO, lr_scheduler_type=LR_SCHEDULER, weight_decay=WEIGHT_DECAY, optim="adamw_8bit",
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=5,
        eval_strategy="steps", eval_steps=EVAL_EVERY_STEPS, save_strategy="no", seed=SEED,
        output_dir=os.path.join(RUN_DIR, "checkpoints"), report_to="none", packing=False, dataset_num_proc=2,
        **{_len_key: MAX_SEQ_LENGTH},
    )
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=monitor_ds, args=args)
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
    labels = tokenizer.decode([t for t in trainer.train_dataset[0]["labels"] if t != -100])
    assert "<|im_start|>user" not in labels and "<think>" not in labels, "prompt not masked"
    print("Loss starts at:", repr(labels[:80]))

    _t0 = time.time()
    trainer.train()
    TRAIN_MINUTES = round((time.time() - _t0) / 60, 2)
    PEAK_VRAM_GB = round(torch.cuda.max_memory_reserved() / 1e9, 2)
    TOKENS_PER_SEC = round(TRAIN_TOKENS * EPOCHS / (TRAIN_MINUTES * 60))
    log = trainer.state.log_history
    json.dump(log, open(os.path.join(RUN_DIR, "train_log.json"), "w"), indent=2)
    print(f"Train {TRAIN_MINUTES} min | {TOKENS_PER_SEC} tokens/s | peak VRAM {PEAK_VRAM_GB} GB")

    import matplotlib.pyplot as plt
    tr = [(l["step"], l["loss"]) for l in log if "loss" in l]
    ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
    plt.figure(figsize=(7, 4)); plt.plot(*zip(*tr), label="train loss")
    if ev: plt.plot(*zip(*ev), "o-", label="monitor (val) loss")
    plt.xlabel("step"); plt.ylabel("loss"); plt.title(RUN_NAME); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(os.path.join(RUN_DIR, "loss_curve.png"), dpi=120, bbox_inches="tight"); plt.show()
else:
    print(f"{RUN_MODE} mode: no training")

## 7. Evaluate (or distill)

**Distill mode (self-distillation, a.k.a. rejection-sampling fine-tuning):** the base model reasons well but writes long answers;
the fine-tuned model is fast but terse. Here the base model answers the *training* questions (never evaluation ones), our own scorers
check each answer, and only reasoning that reached the correct answer is kept. Its final line is rewritten to the gold answer in canonical
form; rejected questions keep their template target, so v2-A and v2-B train on identical questions and differ only in target style.

**What gets scored:** the fixed 600 validation QA questions (300 FinQA + 300 TAT-QA, never trained on) and the 30 synthetic test examples,
with the same scorers in both modes.

**How the prompts differ:**

| Model | Prompt |
|---|---|
| Fine-tuned | Exactly the training prompt (system + question) |
| Base, zero-shot | The same prompt. Measures what the untouched model does with our instructions. |
| Base, few-shot | The same, preceded by 2 worked examples (1 per synthetic task) as earlier chat turns, chosen from the **training** split only |
| `*_rules` variants | The same, plus the QA answer conventions spelled out (scale words, `; ` between items, % with 2+ decimals), so the base model is not penalised for conventions only the fine-tuned model learned |

**Three fairness rules for the base model** (a baseline only means something if it is given a fair chance):
1. A higher answer budget (`MAX_NEW_TOKENS_BASE`), because untrained models write longer answers. A truncated answer would be scored as wrong for the wrong reason.
2. Both **strict** (needs the `Answer:` line) and **lenient** (falls back to the last number) scores are reported, so format failures are visible separately from reasoning failures.
3. Generation stops once an `Answer:` line is complete. It saves GPU time; the only text cut is what comes after the answer, which the scorer never reads.

Prompts are sorted by length before batching, so each batch pads to a similar length (less wasted compute).

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
STOP_IDS = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id


class StopAfterAnswerLine(StoppingCriteria):
    """Per-sequence stop: a sequence is done once its new text has a complete 'Answer: ...' line."""
    def __init__(self, prompt_len):
        self.prompt_len = prompt_len
    def __call__(self, input_ids, scores, **kwargs):
        tails = tokenizer.batch_decode(input_ids[:, self.prompt_len:], skip_special_tokens=True)
        return torch.tensor([answer_line_complete(t) for t in tails], device=input_ids.device)


def zero_shot_prompt(record):
    return to_chatml(build_messages(record))

def few_shot_prompt(record):
    return to_chatml(build_messages(record, SHOTS[source_key(record)]))

def zero_shot_rules_prompt(record):
    return to_chatml(build_messages(record, format_rules=True))

def few_shot_rules_prompt(record):
    return to_chatml(build_messages(record, SHOTS[source_key(record)], format_rules=True))

PROMPTS = {"zero_shot": zero_shot_prompt, "few_shot": few_shot_prompt,
           "zero_shot_rules": zero_shot_rules_prompt, "few_shot_rules": few_shot_rules_prompt}
assert set(BASELINE_VARIANTS) <= set(PROMPTS), f"unknown variant in {BASELINE_VARIANTS}"


def generate(records, prompt_fn, max_new_tokens, batch_size, stop_after_answer, label):
    prompts = [prompt_fn(r) for r in records]
    order = sorted(range(len(records)), key=lambda i: len(prompts[i]))
    out = [None] * len(records); t0 = time.time()
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True).to("cuda")
        prompt_len = enc["input_ids"].shape[1]
        stopping = StoppingCriteriaList([StopAfterAnswerLine(prompt_len)]) if stop_after_answer else None
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=PAD_ID,
                                 stopping_criteria=stopping)
        for i, g in zip(idx, gen):
            new = g[prompt_len:].tolist()
            text = tokenizer.decode(new, skip_special_tokens=True).strip()
            stop = next((j for j, t in enumerate(new) if t in STOP_IDS), None)
            ended = stop is not None or (stop_after_answer and answer_line_complete(text + "\n"))
            n_tokens = (stop + 1) if stop is not None else len([t for t in new if t != PAD_ID])
            out[i] = {"example_id": records[i]["example_id"], "prediction": text, "n_new_tokens": n_tokens,
                      "hit_token_limit": not ended and n_tokens >= max_new_tokens,
                      "prompt_tokens": int(enc["attention_mask"][list(idx).index(i)].sum())}
    minutes = round((time.time() - t0) / 60, 2)
    hits = [p["example_id"] for p in out if p["hit_token_limit"]]
    print(f"{label}: {len(out)} answers in {minutes} min | hit token limit: {len(hits)}")
    return out, minutes, hits


def save_jsonl(name, items):
    with open(os.path.join(RUN_DIR, name), "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")


TATQA_DIR = P("data/external/raw/TAT-QA")
REPORTS = {}

def evaluate(tag, prompt_fn, max_qa, max_synth, batch_size, stop_after_answer, qa_records=None):
    qa_records = qa_eval_records if qa_records is None else qa_records
    qa_preds, qa_min, qa_hits = generate(qa_records, prompt_fn, max_qa, batch_size, stop_after_answer, f"{tag} QA")
    sy_preds, sy_min, sy_hits = generate(synth["test"], prompt_fn, max_synth, batch_size, stop_after_answer,
                                         f"{tag} synthetic")
    save_jsonl(f"{tag}_qa_predictions.jsonl", qa_preds); save_jsonl(f"{tag}_synthetic_predictions.jsonl", sy_preds)
    qa_rep = build_qa_report(qa_records, {p["example_id"]: p["prediction"] for p in qa_preds}, TATQA_DIR)
    sy_rep = build_report(synth["test"], {p["example_id"]: p["prediction"] for p in sy_preds})
    meta = {"run_name": RUN_NAME, "repo_commit": REPO_COMMIT, "model": tag, "stop_after_answer": stop_after_answer,
            "max_new_tokens": {"qa": max_qa, "synthetic": max_synth},
            "qa_token_limit_hits": qa_hits, "synthetic_token_limit_hits": sy_hits,
            "qa_minutes": qa_min, "synthetic_minutes": sy_min}
    if prompt_fn in (few_shot_prompt, few_shot_rules_prompt):
        meta["few_shot_examples"] = {k: [e["example_id"] for e in v] for k, v in SHOTS.items()}
    qa_rep["run"] = meta; sy_rep["run"] = meta
    for name, rep, fmt in [(f"{tag}_qa_report", qa_rep, format_qa_report),
                           (f"{tag}_synthetic_report", sy_rep, format_report)]:
        json.dump(rep, open(os.path.join(RUN_DIR, name + ".json"), "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        open(os.path.join(RUN_DIR, name + ".txt"), "w", encoding="utf-8").write(fmt(rep))
    print(format_qa_report(qa_rep))
    REPORTS[tag] = {"qa": qa_rep, "synthetic": sy_rep, "minutes": round(qa_min + sy_min, 2)}


DISTILL_STATS = None
if RUN_MODE == "train":
    evaluate("finetuned", zero_shot_prompt, MAX_NEW_TOKENS_QA, MAX_NEW_TOKENS_SYNTH, GEN_BATCH_SIZE,
             stop_after_answer=bool(DISTILLED_TARGETS))
elif RUN_MODE == "test":
    evaluate("test", zero_shot_prompt, 1024 if REASONING_STYLE else 384, MAX_NEW_TOKENS_SYNTH, GEN_BATCH_SIZE,
             stop_after_answer=REASONING_STYLE, qa_records=test_records)
elif RUN_MODE == "distill":
    # Self-distillation: base model + format-rules prompt on the mix's TRAINING questions only.
    qa_train = [r for r in train_records if r.get("source_type") in ("finqa", "tatqa")]
    assert not {r["example_id"] for r in qa_train} & eval_ids, "LEAK: distilling on evaluation questions"
    preds, minutes, hits = generate(qa_train, zero_shot_rules_prompt, MAX_NEW_TOKENS_BASE, GEN_BATCH_SIZE,
                                    STOP_AFTER_ANSWER, "distill")
    by_id = {p["example_id"]: p for p in preds}
    rows = build_distilled_targets(qa_train, by_id,
                                   lambda r, text: score_record(r, text, TATQA_DIR)["lenient"]["correct"])
    save_jsonl("distilled_targets_v2_pilot.jsonl", rows)
    from collections import Counter
    DISTILL_STATS = {
        "questions": len(rows), "accepted": sum(r["accepted"] for r in rows), "minutes": minutes,
        "accepted_by_source": dict(Counter(r["source"] for r in rows if r["accepted"])),
        "total_by_source": dict(Counter(r["source"] for r in rows)),
        "rejections": dict(Counter(r["rejection_reason"] for r in rows if not r["accepted"])),
        "accepted_by_answer_type": dict(Counter(r["answer_type"] for r in rows if r["accepted"])),
    }
    print(json.dumps(DISTILL_STATS, indent=2))
else:
    for variant in BASELINE_VARIANTS:
        evaluate(f"base_{variant}", PROMPTS[variant], MAX_NEW_TOKENS_BASE, MAX_NEW_TOKENS_BASE, GEN_BATCH_SIZE,
                 stop_after_answer=STOP_AFTER_ANSWER)

In [ ]:
# Side-by-side summary of everything evaluated in this run
def headline(tag):
    qa, sy = REPORTS[tag]["qa"]["summary"], REPORTS[tag]["synthetic"]["summary"]
    f, t = qa.get("finqa"), qa.get("tatqa")
    return {
        "finqa_strict": f and f["strict"]["accuracy"], "finqa_lenient": f and f["lenient"]["accuracy"],
        "finqa_ci95": f and f["strict"]["ci95"],
        "tatqa_em": t and t["official"]["em"], "tatqa_f1": t and t["official"]["f1"],
        "tatqa_lenient_em": t and t["official"]["lenient_em"], "tatqa_f1_ci95": t and t["official"]["f1_ci95"],
        "answer_line_rate": sum(x["answer_line_rate"] for x in (f, t) if x) / len([x for x in (f, t) if x]),
        "synthetic_numeric_recall": sy["numerical"]["numeric_recall"],
        "synthetic_classification": sy["classification_metrics"]["accuracy"],
        "synthetic_classification_missing": sy["classification_metrics"]["missing_predictions"],
        "eval_minutes": REPORTS[tag]["minutes"],
    }

RESULTS = {tag: headline(tag) for tag in REPORTS}
if not RESULTS:
    print("No evaluation in this mode.")
pct = lambda x: "-" if x is None else f"{x * 100:.1f}%"
rows = [] if not RESULTS else [("FinQA accuracy (strict)", "finqa_strict"), ("FinQA accuracy (lenient)", "finqa_lenient"),
        ("TAT-QA EM (official)", "tatqa_em"), ("TAT-QA F1 (official)", "tatqa_f1"),
        ("TAT-QA EM (lenient)", "tatqa_lenient_em"), ("'Answer:' line rate", "answer_line_rate"),
        ("Synthetic numeric recall", "synthetic_numeric_recall"), ("Synthetic classification", "synthetic_classification")]
if RESULTS:
    print(f"{'':28}" + "".join(f"{t:>22}" for t in RESULTS))
for label, key in rows:
    print(f"{label:28}" + "".join(f"{pct(RESULTS[t][key]):>22}" for t in RESULTS))

## 8. Manifest + save

In [ ]:
import platform, datetime, shutil, zipfile
from importlib.metadata import version, PackageNotFoundError
def v(p):
    try: return version(p)
    except PackageNotFoundError: return None
VERSIONS = {p: v(p) for p in ["unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft", "accelerate",
                              "bitsandbytes", "datasets", "pandas"]}

manifest = {
    "run_name": RUN_NAME, "run_mode": RUN_MODE, "platform": PLATFORM,
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "repo_url": REPO_URL, "repo_commit": REPO_COMMIT, "eval_subset": EVAL_SUBSET, "model_name": MODEL_NAME,
    "evaluation": {"qa_questions": len(qa_eval_records), "synthetic_test": len(synth["test"])},
    "results": RESULTS,
    "environment": {"python": platform.python_version(), "cuda": torch.version.cuda, "gpu": GPU_NAME, "packages": VERSIONS},
}
if RUN_MODE == "train":
    manifest["mix_file"] = MIX_FILE
    manifest["data"] = {"train_examples": len(train_fit), "train_dropped_over_length": TRAIN_DROPPED,
                        "train_tokens": TRAIN_TOKENS, "monitor_examples": len(monitor_fit)}
    manifest["training"] = {"seed": SEED, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
                            "target_modules": TARGET_MODULES, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                            "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "warmup_ratio": WARMUP_RATIO,
                            "lr_scheduler": LR_SCHEDULER, "weight_decay": WEIGHT_DECAY,
                            "max_seq_length": MAX_SEQ_LENGTH, "loss_masking": "assistant only",
                            "precision": "bf16" if is_bfloat16_supported() else "fp16"}
    manifest["timing"] = {"train_minutes": TRAIN_MINUTES, "tokens_per_second": TOKENS_PER_SEC,
                          "peak_vram_gb": PEAK_VRAM_GB}
    model.save_pretrained(os.path.join(RUN_DIR, "adapter")); tokenizer.save_pretrained(os.path.join(RUN_DIR, "adapter"))
    if DISTILLED_INFO:
        manifest["distilled_targets"] = DISTILLED_INFO
elif RUN_MODE == "test":
    manifest["test"] = {"adapter_run": ADAPTER_RUN, "adapter_dir": ADAPTER_DIR, "adapter_config": adapter_cfg,
                        "sources": TEST_SOURCES, "questions": len(test_records), "reasoning_style": REASONING_STYLE}
    manifest["timing"] = {"peak_vram_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2)}
elif RUN_MODE == "distill":
    manifest["distill"] = {**DISTILL_STATS, "prompt": "zero_shot_rules", "max_new_tokens": MAX_NEW_TOKENS_BASE,
                           "acceptance": "lenient answer correct by the evaluation scorers + quality checks"}
    manifest["timing"] = {"peak_vram_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2)}
else:
    manifest["baseline"] = {"variants": BASELINE_VARIANTS, "max_new_tokens": MAX_NEW_TOKENS_BASE,
                            "max_seq_length": BASE_MAX_SEQ_LENGTH, "stop_after_answer": STOP_AFTER_ANSWER,
                            "few_shot_examples": {k: [e["example_id"] for e in v] for k, v in SHOTS.items()}}
    manifest["timing"] = {"peak_vram_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2)}

json.dump(manifest, open(os.path.join(RUN_DIR, "run_manifest.json"), "w"), indent=2)
open(os.path.join(RUN_DIR, "requirements-kaggle.txt" if IN_KAGGLE else "requirements-colab.txt"), "w").write(
    "\n".join(f"{p}=={x}" for p, x in VERSIONS.items() if x) + "\n")
shutil.rmtree(os.path.join(RUN_DIR, "checkpoints"), ignore_errors=True)

results_zip = os.path.join(WORK_DIR, RUN_NAME + "-results.zip")   # small: everything except the adapter
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(RUN_DIR)):
        if f != "adapter":
            z.write(os.path.join(RUN_DIR, f), f)
print(json.dumps(manifest["timing"], indent=2))
print("Results zip (no adapter):", results_zip)

if IN_COLAB and COLAB_USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(RUN_DIR, f"/content/drive/MyDrive/finexpert/runs/{RUN_NAME}", dirs_exist_ok=True)
elif IN_KAGGLE:
    print("Kaggle: everything in /kaggle/working is kept as this version's Output. Download the results zip from there.")

## Reading the results
- **Compare only like with like:** every number here is on the same fixed validation questions, so `base_zero_shot`,
  `base_few_shot` and `finetuned` rows are directly comparable. Quote the 95% CI.
- **Strict vs lenient gap:** large for a base model means it reasons but ignores the `Answer:` format; near zero for the fine-tuned model.
- **Few-shot vs zero-shot:** how much showing two examples helps. Fine-tuning must beat **few-shot**, not just zero-shot, to be worth it.
- **Token-limit hits** should be ~0; otherwise the scores reflect cut-off answers.
- The official **test** sets are not touched here; they are scored once, for the final models.